#

In [1]:
from transformers import logging
logging.set_verbosity_error()  # Mitteilungen muten, damit Dokuemnt ordentlich bleibt

In [2]:
import pandas as pd
from transformers import pipeline
import json

daten = pd.read_csv("../data/example_dataset.csv")
datenliste = list(daten["text"])

In [3]:
generate = pipeline("text-generation", model="meta-llama/Meta-Llama-3.1-8B-Instruct")
ergebnis = []
for text in datenliste:
    instructions = [
        {"role": "system", 
         "content": "Du bist ein trainierter Assistent für Inhaltsanalyse, der die allgemeine Stimmung beziehungsweise Tonalität von Texten analysiert. Antworte immer präzise im JSON-Format mit sentiment (positiv, neutral, negativ) und reasoning (eine Begründung auf Deutsch). Gib ausschließlich den JSON-Response, beginnend mit '{' und endend mit '}', mit diesen zwei Parametern zurück"},
        {"role": "user", "content": text}]
    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    ergebnis.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 68.37it/s]

In [4]:
def parse_response(response):
    parsed = json.loads(response)
    return parsed['sentiment'], parsed['reasoning']

parsed_data = [(entry['text'], *parse_response(entry['response'])) for entry in ergebnis]
parsed_data = pd.DataFrame(parsed_data, columns=['text', 'sentiment', 'reasoning'])
pd.set_option('display.width', 1000)
print(parsed_data)

                                                text sentiment                                          reasoning
0  Die orthodoxe Gemeinde feiert heute #Ostern. I...   positiv  Der Text wünscht ein 'frohes und gesegnetes Fe...
1  Vielen Dank Gregor Rutz für die Unterstützung ...   positiv  Der Text ist positiv, da er jemanden (Gregor R...
2  Wird ja immer schlimmer mit den Intoleranten d...   negativ  Der Text beschreibt eine negative Entwicklung,...
3  Liebe Junge, geht wählen. Kann ja nicht sein, ...   negativ  Der Text enthält eine Aufforderung zum Handeln...
4  ‘— Das war unser EU-Wahl-Abschluss der SP– mit...   positiv  Der Text enthält ein Ausrufezeichen, das auf e...
5  Aufschlussreicher Blick hinter die Mauern! Tol...   positiv  Der Text enthält positive Wörter wie 'Tolle', ...
6  Denn ein Hard-Brexit würde der #EU, aber noch ...   negativ  Der Text besagt, dass ein Hard-Brexit sowohl d...
7  Familienbonus: Es ist erstaunlich wie viele Me...   negativ  Die Verwendung des Worte